In [1]:
import fasttext
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer


/data/shubham/miniconda/envs/namo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("text_files/Hindi_sample.txt", 'r', encoding='utf-8') as file:
    hindi_text = file.read()

In [3]:
file_name = "Hindi_sample.txt"

In [4]:
print(hindi_text[:500])  # Print the first 500 characters to verify the content

कालनिर्णय	: जिस दिन सूर्यास्त के बाद एक घड़ी अधिक तक अमावस्या रहे उस दिन दीपावली मानी जानी चाहिये। दोनों दिन सायंकाल के समय हो तो दूसरे दिन और केवल पहले दिन ही सायंकाल के समय अमा-बस्या हो तो पहले दिन दीपावली मानना उचित है। यदि दोनों ही दिन सायंकाल के समय अमावस्या न हो तो कुछ लोगों के मत से पहले दिन और कुछ लोगों के मत से दूसरे दिन दीपावली मानना चाहिए। इस उत्सव के साथ स्वाति नक्षत्र का योग प्रशस्त माना गया है। दीपावली के साथ धनत्रयोदशी और नरकचतुर्दशी के उत्सव भी हैं।
विधि : इस उत्सव में त्रयोदशी क


step 1: Identify sanskrit sentences.

step 2: replace them with links.

step 3: do semantic chunking of the resulting text

### Step 1 & 2

In [5]:
model = fasttext.load_model('model/indiclid-ftn/model_baseline_native.bin')

In [6]:
def check_lang(text):
    labels, probs = model.predict(text, k=1)
    return labels[0].replace("__label__", ""), probs[0]

# Testing with Devanagari
print(f"Sanskrit Test: {check_lang('अहं गच्छामि')}")
print(f"Hindi Test: {check_lang('मैं जा रहा हूँ')}")

Sanskrit Test: ('san_Deva', 1.0000394582748413)
Hindi Test: ('mai_Deva', 1.000040054321289)


In [7]:
parts = re.split(r'([।:॥\-]+)', hindi_text)
sentences = [
    parts[i].strip() + parts[i + 1]
    for i in range(0, len(parts) - 1, 2)
    if parts[i].strip()
]
print(f"Total sentences: {len(sentences)}")

Total sentences: 228


In [8]:
sans_dict = {}
counter = 0
full_text = ""
for sentence in sentences:
    if '\n' in sentence:
        sentence = sentence.replace('\n', ' ')
    # print(f"Sentence: {sentence.strip()} - Language: {check_lang(sentence.strip())}")
    lang = check_lang(sentence.strip())[0]
    if lang == 'san_Deva' or lang == 'san_Latn':
        sans_dict[file_name.split('.')[0]+'_'+str(counter)] = sentence.strip()
        full_text += f"<sanskrit_chunk id='{file_name.split('.')[0]}_{counter}'>"
        counter += 1
    else:
        full_text += sentence.strip() + " "



In [9]:
print(full_text)

<sanskrit_chunk id='Hindi_sample_0'>जिस दिन सूर्यास्त के बाद एक घड़ी अधिक तक अमावस्या रहे उस दिन दीपावली मानी जानी चाहिये। दोनों दिन सायंकाल के समय हो तो दूसरे दिन और केवल पहले दिन ही सायंकाल के समय अमा- बस्या हो तो पहले दिन दीपावली मानना उचित है। यदि दोनों ही दिन सायंकाल के समय अमावस्या न हो तो कुछ लोगों के मत से पहले दिन और कुछ लोगों के मत से दूसरे दिन दीपावली मानना चाहिए। इस उत्सव के साथ स्वाति नक्षत्र का योग प्रशस्त माना गया है। दीपावली के साथ धनत्रयोदशी और नरकचतुर्दशी के उत्सव भी हैं। विधि: इस उत्सव में त्रयोदशी के सायंकाल के समय यमदीपदान, चतुर्दशी के प्रातःकाल अभ्यङ्ग स्नान और अमावस्या के दिन सायंकाल के समय दीपावली और लक्ष्मीपूजन होते हैं। काल- <sanskrit_chunk id='Hindi_sample_1'>ऋतुओं के वर्णन में ऊपर बार- चार बताया जा चुका है कि भारतवर्ष की सर्वोत्तम ऋतुएँ दो ही हैं- <sanskrit_chunk id='Hindi_sample_2'>इनमें से वसन्त की शोभा भारतवर्ष के जलप्राय और वृक्षावलियों से शोभित प्रदेशों में ही उल्लसित होती है, किन्तु शरद् ऋतु भारतवर्ष के कोने- कोने में- चाहे वह सरुभूमि हो अथवा जलप्लावित

### Step 3

In [10]:
def overlap_tag_splitter(full_text):
    # This regex catches: 
    # 1. The whole Sanskrit tag
    # 2. Hindi punctuation marks
    delimiter_pattern = r"(<sanskrit_chunk id='[^']*'>|[।॥\n]+)"
    
    # We use capturing groups so re.split keeps the delimiters in the list
    raw_parts = re.split(delimiter_pattern, full_text)
    
    final_sentences = []
    current_segment = ""

    for part in raw_parts:
        if not part: continue
        
        # TYPE 1: The Sanskrit Tag (The Pivot)
        if part.startswith("<sanskrit_chunk"):
            # A: If we have text before this tag, close that sentence with the tag
            if current_segment.strip():
                final_sentences.append(current_segment + part)
            
            # B: Immediately start the NEXT sentence with the same tag
            current_segment = part
            
        # TYPE 2: Hindi Punctuation (The Hard Stop)
        elif re.match(r'[।॥\n]+', part):
            if current_segment.strip():
                final_sentences.append(current_segment + part)
            current_segment = ""
            
        # TYPE 3: Regular Hindi Text
        else:
            current_segment += part

    # Catch any remaining text at the end
    if current_segment.strip():
        final_sentences.append(current_segment)
        
    return final_sentences

In [11]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import numpy as np

# 1. Setup Model and Tokenizer
model_name = "BAAI/bge-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def get_bge_embeddings(sentences, batch_size=32):
    """
    Generates BGE-M3 embeddings using the transformers library.
    Bypasses multiprocessing bugs by running on the main process.
    """
    all_embeddings = []
    
    # Process in batches for memory efficiency
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        
        # Tokenize
        encoded = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
            
            # BGE-M3 uses the [CLS] token (index 0) for its dense representation
            # Shape: (batch_size, 1024)
            embeddings = outputs.last_hidden_state[:, 0]
            
            # Unit Normalization (L2) - This is CRITICAL for BGE-M3 
            # to spread out the 'Narrow Cone' of Hindi embeddings.
            embeddings = F.normalize(embeddings, p=2, dim=1)
            
            all_embeddings.append(embeddings.cpu().numpy())
            
    return np.concatenate(all_embeddings, axis=0)

# --- Usage ---
# embeddings = get_bge_embeddings(clean_for_model, batch_size=32)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 59702.68it/s]


In [12]:
# 1. Generate the overlapping sentences
sentences = overlap_tag_splitter(full_text)

# 2. Create a 'Masked' version for the embedding model
# We remove the tags so the model only focuses on the actual Hindi meaning
clean_sentences = [
    re.sub(r"<sanskrit_chunk id='[^']*'>", "", s).strip() 
    for s in sentences
]

# 3. Filter out any fragments that became empty after stripping (safety check)
# BGE-M3 will throw an error if you pass it an empty string
valid_indices = [i for i, s in enumerate(clean_sentences) if len(s) > 0]
clean_for_model = [clean_sentences[i] for i in valid_indices]
original_for_db = [sentences[i] for i in valid_indices]

In [13]:
# 4. Generate Embeddings
embeddings = get_bge_embeddings(clean_for_model, batch_size=32)

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

def generate_semantic_chunks(
    embeddings, 
    original_sentences, 
    threshold=0.8, 
    threshold_step=0.01, 
    alpha=0.3, 
    lookahead_size=2
):
    """
    Groups sentences into semantic chunks using a decaying running average 
    and a lookahead window.
    """
    if len(embeddings) == 0:
        return []

    chunks = []
    current_chunk_text = []
    running_avg_vec = None
    
    i = 0
    while i < len(embeddings):
        curr_vec = embeddings[i]
        curr_text = original_sentences[i]

        # 1. Initialize a new chunk if necessary
        if running_avg_vec is None:
            running_avg_vec = curr_vec
            current_chunk_text.append(curr_text)
            i += 1
            continue

        # 2. Calculate the Lookahead Window Average
        # We look at the next 'lookahead_size' sentences to see where the topic is going
        lookahead_end = min(i + lookahead_size, len(embeddings))
        lookahead_vecs = embeddings[i:lookahead_end]
        lookahead_avg = np.mean(lookahead_vecs, axis=0)

        # 3. Calculate Dynamic Threshold
        # As the chunk gets longer, we increase the threshold to force a split
        # This prevents the 'Dilution'/Inertia problem
        current_threshold = threshold + (len(current_chunk_text) * threshold_step)
        # Cap the threshold at 0.98 to avoid impossible matches
        current_threshold = min(current_threshold, 0.98)

        # 4. Check Similarity
        sim = cosine_similarity(
            running_avg_vec.reshape(1, -1), 
            lookahead_avg.reshape(1, -1)
        )[0][0]

        if sim >= current_threshold:
            # Topic continues: Update the running average (EWMA)
            # This gives more weight to the most recent sentence (Recency Bias)
            running_avg_vec = (1 - alpha) * running_avg_vec + alpha * curr_vec
            current_chunk_text.append(curr_text)
            i += 1
        else:
            # Topic shift detected: Close current chunk and start new one
            chunks.append(" ".join(current_chunk_text))
            current_chunk_text = [curr_text]
            running_avg_vec = curr_vec
            i += 1

    # Catch the last chunk
    if current_chunk_text:
        chunks.append(" ".join(current_chunk_text))

    return chunks

# %% Execution
# Using your already generated 'embeddings' and 'original_for_db'
final_chunks = generate_semantic_chunks(
    embeddings=embeddings,
    original_sentences=original_for_db,
    threshold=0.52,          # Base similarity starting point
    threshold_step=0.005,    # Tighten threshold by 0.005 for every sentence in chunk
    alpha=0.3,               # Give 30% weight to the newest sentence in the centroid
    lookahead_size=2         # Look 2 sentences ahead to filter out 'bridge' noise
)

print(f"Generated {len(final_chunks)} semantic chunks from {len(original_for_db)} sentences.")
for idx, chunk in enumerate(final_chunks):
    print(f"\n--- Chunk {idx+1} ---\n{chunk[:200]}...")

Generated 11 semantic chunks from 134 sentences.

--- Chunk 1 ---
<sanskrit_chunk id='Hindi_sample_0'>जिस दिन सूर्यास्त के बाद एक घड़ी अधिक तक अमावस्या रहे उस दिन दीपावली मानी जानी चाहिये।  दोनों दिन सायंकाल के समय हो तो दूसरे दिन और केवल पहले दिन ही सायंकाल के समय ...

--- Chunk 2 ---
 काल- <sanskrit_chunk id='Hindi_sample_1'>...

--- Chunk 3 ---
<sanskrit_chunk id='Hindi_sample_1'>ऋतुओं के वर्णन में ऊपर बार- चार बताया जा चुका है कि भारतवर्ष की सर्वोत्तम ऋतुएँ दो ही हैं- <sanskrit_chunk id='Hindi_sample_2'> <sanskrit_chunk id='Hindi_sample_2'>...

--- Chunk 4 ---
 विधि- <sanskrit_chunk id='Hindi_sample_3'>...

--- Chunk 5 ---
<sanskrit_chunk id='Hindi_sample_3'>भारतवर्ष के चार प्रधान राष्ट्रीय त्यौहारों में से, जैसे कि- रक्षाबंधन के प्रकरण में लिखा जा चुका है, यह त्यौहार- भी एक है।  इस त्यौहार में लक्ष्मी पूजन की मुख्यता क...

--- Chunk 6 ---
 भारतवर्ष का यह उत्सव सचमुच लक्ष्मी के आधिभौतिक, आध्या- त्मिक और आधिदैविक तीनों स्वरूपों का उल्लासमय प्राकट्य करनेवाला है।  लक्ष्मी का आधिभौतिक-

In [15]:
for chunk in final_chunks:
    for sentence in chunk.split(r'[।॥\n]+'):
        print(sentence.strip())
    print("\n---\n")


<sanskrit_chunk id='Hindi_sample_0'>जिस दिन सूर्यास्त के बाद एक घड़ी अधिक तक अमावस्या रहे उस दिन दीपावली मानी जानी चाहिये।  दोनों दिन सायंकाल के समय हो तो दूसरे दिन और केवल पहले दिन ही सायंकाल के समय अमा- बस्या हो तो पहले दिन दीपावली मानना उचित है।  यदि दोनों ही दिन सायंकाल के समय अमावस्या न हो तो कुछ लोगों के मत से पहले दिन और कुछ लोगों के मत से दूसरे दिन दीपावली मानना चाहिए।  इस उत्सव के साथ स्वाति नक्षत्र का योग प्रशस्त माना गया है।  दीपावली के साथ धनत्रयोदशी और नरकचतुर्दशी के उत्सव भी हैं।  विधि: इस उत्सव में त्रयोदशी के सायंकाल के समय यमदीपदान, चतुर्दशी के प्रातःकाल अभ्यङ्ग स्नान और अमावस्या के दिन सायंकाल के समय दीपावली और लक्ष्मीपूजन होते हैं।

---

काल- <sanskrit_chunk id='Hindi_sample_1'>

---

<sanskrit_chunk id='Hindi_sample_1'>ऋतुओं के वर्णन में ऊपर बार- चार बताया जा चुका है कि भारतवर्ष की सर्वोत्तम ऋतुएँ दो ही हैं- <sanskrit_chunk id='Hindi_sample_2'> <sanskrit_chunk id='Hindi_sample_2'>इनमें से वसन्त की शोभा भारतवर्ष के जलप्राय और वृक्षावलियों से शोभित प्रदेशों में ही उल्

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

# Check similarity between every 2 adjacent sentences
sims = []
for i in range(len(embeddings) - 1):
    s = cosine_similarity(embeddings[i].reshape(1,-1), embeddings[i+1].reshape(1,-1))[0][0]
    sims.append(s)

print(f"Min Sim: {min(sims):.4f}")
print(f"Max Sim: {max(sims):.4f}")
print(f"Avg Sim: {np.mean(sims):.4f}")

Min Sim: 0.2355
Max Sim: 0.8634
Avg Sim: 0.5047


In [1]:
import json

with open("text_files/book1.json", 'r', encoding='utf-8') as file:
    data = json.load(file)

In [2]:
chapters = [ele['Heading'] + ": " + ele['Paragraph Text'] for ele in data]

In [6]:
chapters[2]

"हनुमज्जयन्ती/हनुमान् जयन्ती: हनुमज्जयन्ती के समय के विषय में मत भेद है- उत्सवसिन्धु, व्रतरत्नाकर और वाल्मीकीयरामायण से कार्तिककृष्णा चतुर्दशी' सिद्ध होता है, किन्तु कुछ विद्वान् चैत्रशुक्ला पूर्णिमा मानते हैं। लोक में भी चैत्रशुक्ला पूर्णिमा ही अधिक प्रचलित है। ऐसी दशा में निश्चित निर्णय असम्भव है। जो लोग जैसा मानने हैं, मानते रहें। यह तिथि सायंकालव्यापिनी लेनी चाहिए, क्योंकि हनुमान् जी का जन्म रात्रि में माना जाता है।जयन्ती मनाने की विधि रामनवमी के प्रसङ्ग में लिखी जा चुकी है। तद्\u200cनुसार ही व्रत, उपवास और पत्रामृतस्नानादि इस दिन भी करना चाहिए। शृङ्गार में तैल सिन्दूर आदि और नैवेद्य में चना (भीगा हुआ अथवा भुना हुआ) गुड़ और बेसन के लड्डू अथवा बूँदी (मोतीचूर) के लड्डू अवश्य रहने चाहिए । हनुमान् जी बलवानों में प्रधान माने जाते हैं। मल्लविद्याविदों के तो वे इष्टदेव ही हैं। अतः उस दिन व्यायामप्रदर्शन अवश्य होना चाहिए । ऊपर लिखा जा चुका है कि हनुमज्जयन्ती के विषय में मतभेद है, अतः कालविज्ञान पर कुछ नहीं लिखा जा सकता । पञ्चामृत के गुण तो रामनवमी और जन्माष्टमी के प्रसंग में देखिए। शृङ्गार

In [5]:
for i, chapter in enumerate(chapters):
    with open(f"text_files/chapter_{i}.txt", 'w', encoding='utf-8') as file:
        file.write(chapter.strip())

In [5]:
from transformers import AutoTokenizer

tokenizer_model = "google/gemma-4-26B-A4B-it"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)

for chapter in chapters:
    tokens = tokenizer(chapter, truncation=False)['input_ids']
    print(f"Chapter Tokens: {len(tokens)}")

/data/shubham/miniconda/envs/namo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chapter Tokens: 3912
Chapter Tokens: 5916
Chapter Tokens: 665
Chapter Tokens: 2149
Chapter Tokens: 3015
Chapter Tokens: 2223
Chapter Tokens: 1384
Chapter Tokens: 2010
Chapter Tokens: 3951
Chapter Tokens: 5256
Chapter Tokens: 4894
Chapter Tokens: 689
Chapter Tokens: 5864
Chapter Tokens: 5055
Chapter Tokens: 2554
Chapter Tokens: 5266
Chapter Tokens: 4199
Chapter Tokens: 3475
Chapter Tokens: 1514
Chapter Tokens: 771
Chapter Tokens: 1611
Chapter Tokens: 9504
Chapter Tokens: 4236
Chapter Tokens: 6460
Chapter Tokens: 5853
